# Statistical Inference and Regression

This notebook sits between EDA and ML modelling. The main question is:

**What factors are associated with track popularity in this playlist, and do different groups show meaningful differences in popularity?**

The notebook focuses on:

- confidence intervals for popularity
- group differences in popularity
- multiple linear regression with model diagnostics
- logistic regression for high-popularity tracks
- optional Poisson regression for artist-level track counts

## 1. Setup

In [1]:
import ast
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan, linear_reset
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor

alt.data_transformers.disable_max_rows()
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

In [2]:
def find_data_dir():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        notebook_data = candidate / "notebooks" / "data"
        local_data = candidate / "data"
        if notebook_data.exists():
            return notebook_data
        if candidate.name == "notebooks" and local_data.exists():
            return local_data
    return Path("data")


DATA_DIR = find_data_dir()
DATA_PATH = DATA_DIR / "df.csv"
df = pd.read_csv(DATA_PATH)

df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["added_at"] = pd.to_datetime(df["added_at"], errors="coerce", utc=True)
df["release_year"] = df["release_date"].dt.year

df.shape, df.head()

((238, 24),
                  track_id          track_name artist_names  \
 0  0VGPId3riSeBV5FTRn850F                需要人陪  Leehom Wang   
 1  6aF2RVTPJSJCM1MpUjne5X                落葉歸根  Leehom Wang   
 2  68RtalgSqaCbeZN42QeMS0                 花田錯  Leehom Wang   
 3  3agtg0x11wPvLIWkYR39nZ  Somewhere I Belong  Linkin Park   
 4  57BrRMwf9LrcmuOsyGilwr            Crawling  Linkin Park   
 
                artist_ids main_artist_name          main_artist_id  \
 0  2F5W6Rsxwzg0plQ0w8dSyt      Leehom Wang  2F5W6Rsxwzg0plQ0w8dSyt   
 1  2F5W6Rsxwzg0plQ0w8dSyt      Leehom Wang  2F5W6Rsxwzg0plQ0w8dSyt   
 2  2F5W6Rsxwzg0plQ0w8dSyt      Leehom Wang  2F5W6Rsxwzg0plQ0w8dSyt   
 3  6XyY86QOPPrYVGvF9ch6wz      Linkin Park  6XyY86QOPPrYVGvF9ch6wz   
 4  6XyY86QOPPrYVGvF9ch6wz      Linkin Park  6XyY86QOPPrYVGvF9ch6wz   
 
                  album_id                     album_name release_date  \
 0  5lpy0FxmneKqFb4zeoiSHM                          十八般武藝   2010-08-12   
 1  1lxlQW182pklrfpD93faq1      

## 2. Research Questions

1. Do different language/script groups have different popularity distributions?
2. Do release decades differ in popularity?
3. Which numeric variables are associated with popularity?
4. After controlling for artist popularity, followers, release year, duration, and explicitness, does language/script still explain popularity?
5. Can the same variables help explain whether a track is above-median popularity?
6. Optional count model: are more popular artists represented more often in this playlist?

## 3. Data Preparation

In [3]:
def parse_list_like(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

analysis_df = df.copy()
analysis_df["language_group"] = analysis_df["song_title_language"].fillna("Unknown")

# Keep the original language/script groups for EDA and tests, but combine sparse
# groups for regression so diagnostics are not dominated by one-off categories.
language_counts = analysis_df["language_group"].value_counts()
common_languages = language_counts[language_counts >= 10].index
analysis_df["language_model_group"] = np.where(
    analysis_df["language_group"].isin(common_languages),
    analysis_df["language_group"],
    "Other/Mixed",
)

analysis_df["explicit_int"] = analysis_df["explicit"].astype(int)
analysis_df["log_artist_followers"] = np.log1p(analysis_df["artist_followers"])
analysis_df["release_decade"] = (
    (analysis_df["release_year"] // 10 * 10)
    .astype("Int64")
    .astype(str)
    .str.replace("<NA>", "Unknown", regex=False)
    + "s"
)
analysis_df.loc[analysis_df["release_decade"].eq("Unknowns"), "release_decade"] = "Unknown"
analysis_df["high_popularity"] = (analysis_df["popularity"] >= analysis_df["popularity"].median()).astype(int)

model_cols = [
    "popularity",
    "duration_min",
    "artist_popularity",
    "artist_followers",
    "log_artist_followers",
    "release_year",
    "explicit_int",
    "language_group",
    "language_model_group",
    "release_decade",
    "high_popularity",
]
analysis_df = analysis_df.dropna(subset=model_cols)

analysis_df[model_cols].head()

,popularity,duration_min,artist_popularity,artist_followers,log_artist_followers,release_year,explicit_int,language_group,language_model_group,release_decade,high_popularity
0,50,4.185767,57,1192346,13.991434,2010.0,0,Chinese-script title,Chinese-script title,2010s,0
1,51,5.253767,57,1192346,13.991434,2007.0,0,Chinese-script title,Chinese-script title,2000s,1
2,50,3.799333,57,1192346,13.991434,2005.0,0,Chinese-script title,Chinese-script title,2000s,0
3,81,3.565550,89,34542635,17.357705,2003.0,0,Latin-script title,Latin-script title,2000s,1
4,84,3.482667,89,34542635,17.357705,2000.0,0,Latin-script title,Latin-script title,2000s,1


In [4]:
analysis_df[model_cols].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
popularity,232.0,NaN,NaN,NaN,49.681034,21.967548,0.0,37.75,50.5,65.0,92.0
duration_min,232.0,NaN,NaN,NaN,3.790937,0.640641,2.05665,3.351554,3.781908,4.205483,6.35555
artist_popularity,232.0,NaN,NaN,NaN,61.25,19.721601,12.0,51.75,60.0,77.0,96.0
artist_followers,232.0,NaN,NaN,NaN,10534790.081897,19501423.334932,258.0,207820.5,1270568.0,10585263.5,109797882.0
log_artist_followers,232.0,NaN,NaN,NaN,13.938849,2.8358,5.556828,12.244386,14.054975,16.174635,18.514152
release_year,232.0,NaN,NaN,NaN,2011.646552,6.6388,1997.0,2007.0,2012.0,2017.0,2026.0
explicit_int,232.0,NaN,NaN,NaN,0.043103,0.203529,0.0,0.0,0.0,0.0,1.0
language_group,232,4,Latin-script title,152,NaN,NaN,NaN,NaN,NaN,NaN,NaN
language_model_group,232,3,Latin-script title,152,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_decade,232,4,2010s,113,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 4. Confidence Intervals for Popularity

These intervals describe uncertainty in the mean popularity score. They are useful descriptive summaries before formal hypothesis testing.

In [5]:
def mean_ci(x, confidence=0.95):
    x = pd.Series(x).dropna().astype(float)
    n = len(x)
    mean = x.mean()
    if n < 2:
        return pd.Series({"n": n, "mean": mean, "ci_low": np.nan, "ci_high": np.nan})
    se = stats.sem(x)
    margin = se * stats.t.ppf((1 + confidence) / 2, n - 1)
    return pd.Series({"n": n, "mean": mean, "ci_low": mean - margin, "ci_high": mean + margin})

overall_popularity_ci = mean_ci(analysis_df["popularity"])
overall_popularity_ci.round(2).to_frame("overall_popularity")

,overall_popularity
n,232.00
mean,49.68
ci_low,46.84
ci_high,52.52


In [6]:
language_ci = (
    analysis_df.groupby("language_group")["popularity"]
    .apply(mean_ci)
    .unstack()
    .sort_values("n", ascending=False)
    .round(2)
)

language_ci

,n,mean,ci_low,ci_high
language_group,,,,
Latin-script title,152.0,52.74,48.94,56.55
Chinese-script title,71.0,44.59,40.65,48.53
Mixed-script title,8.0,42.00,29.93,54.07
Japanese-script title,1.0,7.00,NaN,NaN


In [7]:
ci_plot_data = language_ci.reset_index()

base = alt.Chart(ci_plot_data).encode(
    y=alt.Y("language_group:N", sort="-x", title="Language/script group"),
)

points = base.mark_point(size=90).encode(
    x=alt.X("mean:Q", title="Mean popularity"),
    tooltip=["language_group:N", "n:Q", alt.Tooltip("mean:Q", format=".2f"), alt.Tooltip("ci_low:Q", format=".2f"), alt.Tooltip("ci_high:Q", format=".2f")],
)

rules = base.mark_rule().encode(
    x="ci_low:Q",
    x2="ci_high:Q",
)

(points + rules).properties(
    title="95% Confidence Intervals for Mean Popularity by Language/Script",
    width=650,
    height=220,
)

alt.LayerChart(...)

## 5. Group Differences in Popularity

We compare popularity across language/script and release-decade groups. Because the playlist is an observational sample, these tests should be interpreted as association and description, not causation.

In [8]:
def cohens_d(x, y):
    x = pd.Series(x).dropna().astype(float)
    y = pd.Series(y).dropna().astype(float)
    nx, ny = len(x), len(y)
    pooled_sd = np.sqrt(((nx - 1) * x.var(ddof=1) + (ny - 1) * y.var(ddof=1)) / (nx + ny - 2))
    return (x.mean() - y.mean()) / pooled_sd

def welch_ci_diff(x, y, confidence=0.95):
    x = pd.Series(x).dropna().astype(float)
    y = pd.Series(y).dropna().astype(float)
    diff = x.mean() - y.mean()
    se = np.sqrt(x.var(ddof=1) / len(x) + y.var(ddof=1) / len(y))
    df_num = (x.var(ddof=1) / len(x) + y.var(ddof=1) / len(y)) ** 2
    df_den = (x.var(ddof=1) ** 2) / ((len(x) ** 2) * (len(x) - 1)) + (y.var(ddof=1) ** 2) / ((len(y) ** 2) * (len(y) - 1))
    dof = df_num / df_den
    margin = stats.t.ppf((1 + confidence) / 2, dof) * se
    return diff, diff - margin, diff + margin

latin = analysis_df.loc[analysis_df["language_group"].eq("Latin-script title"), "popularity"]
chinese = analysis_df.loc[analysis_df["language_group"].eq("Chinese-script title"), "popularity"]

welch = stats.ttest_ind(chinese, latin, equal_var=False)
mann_whitney = stats.mannwhitneyu(chinese, latin, alternative="two-sided")
diff, ci_low, ci_high = welch_ci_diff(chinese, latin)

pd.DataFrame({
    "comparison": ["Chinese-script title - Latin-script title"],
    "mean_diff": [diff],
    "ci_low": [ci_low],
    "ci_high": [ci_high],
    "welch_t": [welch.statistic],
    "welch_p": [welch.pvalue],
    "mann_whitney_u": [mann_whitney.statistic],
    "mann_whitney_p": [mann_whitney.pvalue],
    "cohens_d": [cohens_d(chinese, latin)],
}).round(4)

,comparison,mean_diff,ci_low,ci_high,welch_t,welch_p,mann_whitney_u,mann_whitney_p,cohens_d
0,Chinese-script title - Latin-script title,-8.1519,-13.5952,-2.7085,-2.9543,0.0035,4171.0,0.0064,-0.375


In [9]:
# Multiple pairwise Welch tests across language/script groups with FDR correction.
language_groups = {
    name: group["popularity"].dropna()
    for name, group in analysis_df.groupby("language_group")
    if len(group) >= 5
}

pairwise_results = []
names = list(language_groups)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        g1, g2 = names[i], names[j]
        x, y = language_groups[g1], language_groups[g2]
        test = stats.ttest_ind(x, y, equal_var=False)
        pairwise_results.append({
            "group_1": g1,
            "group_2": g2,
            "n_1": len(x),
            "n_2": len(y),
            "mean_1": x.mean(),
            "mean_2": y.mean(),
            "mean_diff": x.mean() - y.mean(),
            "p_value": test.pvalue,
            "cohens_d": cohens_d(x, y),
        })

pairwise_language = pd.DataFrame(pairwise_results)
if not pairwise_language.empty:
    pairwise_language["p_fdr"] = multipletests(pairwise_language["p_value"], method="fdr_bh")[1]
    pairwise_language = pairwise_language.sort_values("p_fdr")

pairwise_language.round(4)

,group_1,group_2,n_1,n_2,mean_1,mean_2,mean_diff,p_value,cohens_d,p_fdr
0,Chinese-script title,Latin-script title,71,152,44.5915,52.7434,-8.1519,0.0035,-0.3750,0.0106
2,Latin-script title,Mixed-script title,152,8,52.7434,42.0000,10.7434,0.0799,0.4592,0.1199
1,Chinese-script title,Mixed-script title,71,8,44.5915,42.0000,2.5915,0.6468,0.1573,0.6468


In [10]:
# Multi-group tests: language/script and release decade.
def multi_group_tests(data, group_col, response="popularity", min_n=5):
    groups = [g[response].dropna().astype(float) for _, g in data.groupby(group_col) if len(g) >= min_n]
    labels = [name for name, g in data.groupby(group_col) if len(g) >= min_n]
    if len(groups) < 2:
        return pd.Series({"group_col": group_col, "groups_used": labels, "anova_p": np.nan, "kruskal_p": np.nan})
    anova = stats.f_oneway(*groups)
    kruskal = stats.kruskal(*groups)
    grand_mean = data[response].mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
    ss_total = ((data[response] - grand_mean) ** 2).sum()
    eta_squared = ss_between / ss_total
    return pd.Series({
        "group_col": group_col,
        "groups_used": labels,
        "anova_F": anova.statistic,
        "anova_p": anova.pvalue,
        "kruskal_H": kruskal.statistic,
        "kruskal_p": kruskal.pvalue,
        "eta_squared": eta_squared,
    })

pd.DataFrame([
    multi_group_tests(analysis_df, "language_group"),
    multi_group_tests(analysis_df, "release_decade"),
]).round(4)

,group_col,groups_used,anova_F,anova_p,kruskal_H,kruskal_p,eta_squared
0,language_group,"[Chinese-script title, Latin-script title, Mixed-script title]",4.0133,0.0194,9.3364,0.0094,0.0335
1,release_decade,"[1990s, 2000s, 2010s, 2020s]",1.4320,0.2342,3.5651,0.3124,0.0185


## 6. Correlation Analysis

Before fitting regression models, check simple bivariate associations with popularity.

In [11]:
corr_vars = ["release_year", "duration_min", "artist_popularity", "log_artist_followers"]

corr_rows = []
for col in corr_vars:
    pearson = stats.pearsonr(analysis_df[col], analysis_df["popularity"])
    spearman = stats.spearmanr(analysis_df[col], analysis_df["popularity"])
    corr_rows.append({
        "variable": col,
        "pearson_r": pearson.statistic,
        "pearson_p": pearson.pvalue,
        "spearman_rho": spearman.statistic,
        "spearman_p": spearman.pvalue,
    })

pd.DataFrame(corr_rows).round(4)

,variable,pearson_r,pearson_p,spearman_rho,spearman_p
0,release_year,-0.1141,0.0829,-0.0905,0.1693
1,duration_min,-0.0646,0.3272,-0.0849,0.1975
2,artist_popularity,0.5800,0.0000,0.5847,0.0000
3,log_artist_followers,0.5436,0.0000,0.5592,0.0000


In [12]:
scatter_matrix_data = analysis_df[["popularity", "release_year", "duration_min", "artist_popularity", "log_artist_followers", "language_group"]]

alt.Chart(scatter_matrix_data).mark_circle(size=60, opacity=0.65).encode(
    x=alt.X("artist_popularity:Q", title="Artist popularity"),
    y=alt.Y("popularity:Q", title="Track popularity"),
    color=alt.Color("language_group:N", title="Language/script"),
    tooltip=["popularity:Q", "artist_popularity:Q", "log_artist_followers:Q", "language_group:N"],
).properties(
    title="Track Popularity vs Artist Popularity",
    width=650,
    height=360,
)

alt.Chart(...)

## 7. Multiple Linear Regression: Popularity as Response

This model estimates popularity while controlling for multiple predictors. Coefficients for categorical variables are interpreted relative to the reference category selected by the model. Sparse language/script groups are combined into `Other/Mixed` for regression stability.

In [13]:
ols_formula = "popularity ~ release_year + duration_min + artist_popularity + log_artist_followers + explicit_int + C(language_model_group)"
ols_model = smf.ols(ols_formula, data=analysis_df).fit()
ols_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             popularity   R-squared:                       0.370
Model:                            OLS   Adj. R-squared:                  0.350
Method:                 Least Squares   F-statistic:                     18.78
Date:                Sat, 23 May 2026   Prob (F-statistic):           1.25e-19
Time:                        14:32:56   Log-Likelihood:                -991.92
No. Observations:                 232   AIC:                             2000.
Df Residuals:                     224   BIC:                             2027.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
=================================================================================================================
                                                    coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------
Intercept                                       756.0137    384.014      1.969      0.050      -0.728    1512.756
C(language_model_group)[T.Latin-script title]     3.8312      2.779      1.379      0.169      -1.645       9.308
C(language_model_group)[T.Other/Mixed]           -2.8024      6.492     -0.432      0.666     -15.596       9.991
release_year                                     -0.3705      0.189     -1.955      0.052      -0.744       0.003
duration_min                                      1.2018      2.139      0.562      0.575      -3.013       5.417
artist_popularity                                 0.7920      0.210      3.763      0.000       0.377       1.207
log_artist_followers                             -1.1511      1.446     -0.796      0.427      -4.002       1.699
explicit_int                                    -10.7557      5.936     -1.812      0.071     -22.454       0.943
==============================================================================
Omnibus:                       25.864   Durbin-Watson:                   1.962
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               30.945
Skew:                          -0.863   Prob(JB):                     1.91e-07
Kurtosis:                       3.469   Cond. No.                     6.65e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 6.65e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [14]:
# Heteroskedasticity-robust standard errors are often useful for observational data.
ols_model_hc3 = ols_model.get_robustcov_results(cov_type="HC3")
ols_model_hc3.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             popularity   R-squared:                       0.370
Model:                            OLS   Adj. R-squared:                  0.350
Method:                 Least Squares   F-statistic:                     18.05
Date:                Sat, 23 May 2026   Prob (F-statistic):           5.97e-19
Time:                        14:32:56   Log-Likelihood:                -991.92
No. Observations:                 232   AIC:                             2000.
Df Residuals:                     224   BIC:                             2027.
Df Model:                           7                                         
Covariance Type:                  HC3                                         
=================================================================================================================
                                                    coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------
Intercept                                       756.0137    379.459      1.992      0.048       8.247    1503.781
C(language_model_group)[T.Latin-script title]     3.8312      2.512      1.525      0.129      -1.118       8.780
C(language_model_group)[T.Other/Mixed]           -2.8024      6.115     -0.458      0.647     -14.853       9.248
release_year                                     -0.3705      0.187     -1.981      0.049      -0.739      -0.002
duration_min                                      1.2018      1.902      0.632      0.528      -2.545       4.949
artist_popularity                                 0.7920      0.209      3.784      0.000       0.380       1.204
log_artist_followers                             -1.1511      1.403     -0.820      0.413      -3.917       1.615
explicit_int                                    -10.7557      8.789     -1.224      0.222     -28.076       6.564
==============================================================================
Omnibus:                       25.864   Durbin-Watson:                   1.962
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               30.945
Skew:                          -0.863   Prob(JB):                     1.91e-07
Kurtosis:                       3.469   Cond. No.                     6.65e+05
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC3)
[2] The condition number is large, 6.65e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

### 7.1 Linear Regression Diagnostics

Key checks:

- residuals vs fitted: nonlinearity or heteroskedasticity
- Q-Q plot: residual normality
- Breusch-Pagan test: heteroskedasticity
- VIF: multicollinearity
- Cook's distance: influential observations
- RESET test: possible functional-form misspecification

In [15]:
diagnostic_df = analysis_df.copy()
diagnostic_df["fitted"] = ols_model.fittedvalues
diagnostic_df["residual"] = ols_model.resid
diagnostic_df["standardized_residual"] = ols_model.get_influence().resid_studentized_internal

a = alt.Chart(diagnostic_df).mark_circle(size=60, opacity=0.65).encode(
    x=alt.X("fitted:Q", title="Fitted values"),
    y=alt.Y("residual:Q", title="Residuals"),
    tooltip=["track_name:N", "main_artist_name:N", "popularity:Q", "fitted:Q", "residual:Q"],
).properties(title="Residuals vs Fitted", width=315, height=280)

b = alt.Chart(diagnostic_df).mark_circle(size=60, opacity=0.65).encode(
    x=alt.X("fitted:Q", title="Fitted values"),
    y=alt.Y("standardized_residual:Q", title="Standardized residuals"),
    tooltip=["track_name:N", "main_artist_name:N", "standardized_residual:Q"],
).properties(title="Standardized Residuals vs Fitted", width=315, height=280)

a | b

alt.HConcatChart(...)

In [16]:
qq = stats.probplot(ols_model.resid, dist="norm")
qq_df = pd.DataFrame({
    "theoretical_quantile": qq[0][0],
    "ordered_residual": qq[0][1],
})

qq_points = alt.Chart(qq_df).mark_circle(size=45, opacity=0.7).encode(
    x=alt.X("theoretical_quantile:Q", title="Theoretical normal quantiles"),
    y=alt.Y("ordered_residual:Q", title="Ordered residuals"),
    tooltip=["theoretical_quantile:Q", "ordered_residual:Q"],
)

qq_line = alt.Chart(qq_df).mark_line(color="firebrick").encode(
    x="theoretical_quantile:Q",
    y="theoretical_quantile:Q",
)

(qq_points + qq_line).properties(
    title="Q-Q Plot for OLS Residuals",
    width=420,
    height=320,
)

alt.LayerChart(...)

In [17]:
bp_lm, bp_lm_p, bp_f, bp_f_p = het_breuschpagan(ols_model.resid, ols_model.model.exog)
reset = linear_reset(ols_model, power=2, use_f=True)
jarque_bera = stats.jarque_bera(ols_model.resid)
shapiro = stats.shapiro(ols_model.resid) if len(ols_model.resid) <= 5000 else None

diagnostic_tests = pd.DataFrame({
    "test": ["Breusch-Pagan LM", "Breusch-Pagan F", "Jarque-Bera normality", "Shapiro-Wilk normality", "Ramsey RESET"],
    "statistic": [bp_lm, bp_f, jarque_bera.statistic, shapiro.statistic if shapiro else np.nan, reset.fvalue],
    "p_value": [bp_lm_p, bp_f_p, jarque_bera.pvalue, shapiro.pvalue if shapiro else np.nan, reset.pvalue],
})

diagnostic_tests.round(4)

,test,statistic,p_value
0,Breusch-Pagan LM,12.4612,0.0864
1,Breusch-Pagan F,1.8163,0.0852
2,Jarque-Bera normality,30.9448,0.0000
3,Shapiro-Wilk normality,0.9458,0.0000
4,Ramsey RESET,2.5203,0.1138


In [18]:
# Variance inflation factors for the actual OLS design matrix.
exog = pd.DataFrame(ols_model.model.exog, columns=ols_model.model.exog_names)
vif = pd.DataFrame({
    "variable": exog.columns,
    "vif": [variance_inflation_factor(exog.values, i) for i in range(exog.shape[1])],
})

vif.sort_values("vif", ascending=False).round(2)

,variable,vif
0,Intercept,109083.51
5,artist_popularity,12.69
6,log_artist_followers,12.39
4,duration_min,1.38
1,C(language_model_group)[T.Latin-script title],1.29
3,release_year,1.17
2,C(language_model_group)[T.Other/Mixed],1.16
7,explicit_int,1.08


In [19]:
influence = ols_model.get_influence()
influence_df = analysis_df[["track_name", "main_artist_name", "popularity"]].copy()
influence_df["leverage"] = influence.hat_matrix_diag
influence_df["cooks_distance"] = influence.cooks_distance[0]
influence_df["standardized_residual"] = influence.resid_studentized_internal

influence_df.sort_values("cooks_distance", ascending=False).head(10).round(4)

,track_name,main_artist_name,popularity,leverage,cooks_distance,standardized_residual
52,Trouble - Mike Williams Remix,R3HAB,0,0.1049,0.1035,-2.6592
198,きみの て,Every Little Thing,7,0.1136,0.0715,-2.1137
71,Last Night,Morgan Wallen,86,0.1216,0.0631,1.9097
173,Payphone,Maroon 5,88,0.1128,0.0583,1.9151
135,Kiss Fight (feat. gnash) - Ben Phipps Remix,Tülpa & BLANKTS,30,0.1468,0.0491,1.5108
167,Love Yourself,Justin Bieber,13,0.0326,0.0462,-3.3116
50,夜に駆ける,YOASOBI,71,0.1286,0.0330,1.3373
44,If You Could See Me Now,The Script,23,0.1072,0.0295,-1.4020
186,日不落,JOLIN,0,0.0166,0.0166,-2.8011
199,不正常,Jam Hsiao,0,0.0275,0.0163,-2.1486


In [20]:
alt.Chart(influence_df).mark_circle(size=70, opacity=0.7).encode(
    x=alt.X("leverage:Q", title="Leverage"),
    y=alt.Y("standardized_residual:Q", title="Standardized residual"),
    size=alt.Size("cooks_distance:Q", title="Cook's distance"),
    tooltip=["track_name:N", "main_artist_name:N", "popularity:Q", "leverage:Q", "standardized_residual:Q", "cooks_distance:Q"],
).properties(
    title="Influence Diagnostics",
    width=650,
    height=360,
)

alt.Chart(...)

## 8. Logistic Regression: High-Popularity Tracks

Here the response is binary: whether a track is at or above the playlist median popularity. This is a bridge toward ML classification.

As with OLS, sparse language/script groups are combined into `Other/Mixed` before modelling.

In [21]:
logit_formula = "high_popularity ~ release_year + duration_min + artist_popularity + log_artist_followers + explicit_int + C(language_model_group)"
logit_model = smf.logit(logit_formula, data=analysis_df).fit(maxiter=200)
logit_model.summary()

Optimization terminated successfully.
         Current function value: 0.519472
         Iterations 7


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:        high_popularity   No. Observations:                  232
Model:                          Logit   Df Residuals:                      224
Method:                           MLE   Df Model:                            7
Date:                Sat, 23 May 2026   Pseudo R-squ.:                  0.2506
Time:                        14:32:56   Log-Likelihood:                -120.52
converged:                       True   LL-Null:                       -160.81
Covariance Type:            nonrobust   LLR p-value:                 1.046e-14
=================================================================================================================
                                                    coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------
Intercept                                        77.9632     53.249      1.464      0.143     -26.402     182.329
C(language_model_group)[T.Latin-script title]     0.0272      0.364      0.075      0.940      -0.686       0.741
C(language_model_group)[T.Other/Mixed]           -1.4651      1.141     -1.284      0.199      -3.702       0.771
release_year                                     -0.0403      0.026     -1.532      0.125      -0.092       0.011
duration_min                                     -0.0951      0.304     -0.313      0.754      -0.690       0.500
artist_popularity                                 0.0922      0.034      2.740      0.006       0.026       0.158
log_artist_followers                             -0.1503      0.218     -0.690      0.490      -0.578       0.277
explicit_int                                     -2.2148      0.895     -2.475      0.013      -3.969      -0.461
=================================================================================================================
"""

In [22]:
logit_eval = analysis_df.copy()
logit_eval["predicted_probability"] = logit_model.predict(analysis_df)
logit_eval["predicted_class"] = (logit_eval["predicted_probability"] >= 0.5).astype(int)

cm = confusion_matrix(logit_eval["high_popularity"], logit_eval["predicted_class"])
accuracy = accuracy_score(logit_eval["high_popularity"], logit_eval["predicted_class"])
auc = roc_auc_score(logit_eval["high_popularity"], logit_eval["predicted_probability"])

pd.DataFrame({
    "metric": ["accuracy", "roc_auc", "median_popularity_threshold"],
    "value": [accuracy, auc, analysis_df["popularity"].median()],
}).round(4), pd.DataFrame(cm, index=["actual_low", "actual_high"], columns=["pred_low", "pred_high"])

(                        metric    value
 0                     accuracy   0.7026
 1                      roc_auc   0.8084
 2  median_popularity_threshold  50.5000,
              pred_low  pred_high
 actual_low         80         36
 actual_high        33         83)

In [23]:
# Hosmer-Lemeshow style calibration check using deciles of predicted probability.
calibration = logit_eval.copy()
calibration["prob_decile"] = pd.qcut(calibration["predicted_probability"], q=10, duplicates="drop")
calibration_summary = (
    calibration.groupby("prob_decile", observed=True)
    .agg(
        n=("high_popularity", "count"),
        observed_rate=("high_popularity", "mean"),
        mean_predicted_probability=("predicted_probability", "mean"),
    )
    .reset_index()
)

calibration_summary

,prob_decile,n,observed_rate,mean_predicted_probability
0,"(0.0017699999999999999, 0.0893]",24,0.000000,0.056302
1,"(0.0893, 0.206]",23,0.130435,0.144391
2,"(0.206, 0.336]",23,0.347826,0.262392
3,"(0.336, 0.433]",23,0.478261,0.391011
4,"(0.433, 0.506]",23,0.608696,0.475694
5,"(0.506, 0.592]",23,0.391304,0.556717
6,"(0.592, 0.698]",23,0.521739,0.642293
7,"(0.698, 0.784]",23,0.826087,0.741272
8,"(0.784, 0.858]",23,0.782609,0.821165
9,"(0.858, 0.938]",24,0.916667,0.910218


In [24]:
calibration_long = calibration_summary.melt(
    id_vars=["prob_decile", "n"],
    value_vars=["observed_rate", "mean_predicted_probability"],
    var_name="rate_type",
    value_name="rate",
)
calibration_long["prob_decile"] = calibration_long["prob_decile"].astype(str)

alt.Chart(calibration_long).mark_line(point=True).encode(
    x=alt.X("prob_decile:N", title="Predicted probability decile"),
    y=alt.Y("rate:Q", title="Rate", scale=alt.Scale(domain=[0, 1])),
    color=alt.Color("rate_type:N", title="Rate type"),
    tooltip=["prob_decile:N", "rate_type:N", "rate:Q", "n:Q"],
).properties(
    title="Logistic Regression Calibration Check",
    width=650,
    height=320,
)

alt.Chart(...)

## 9. Optional Poisson Regression: Artist Track Counts

Poisson regression is appropriate for count responses. Here the response is how many tracks each main artist contributes to this playlist.

This does **not** model track popularity directly. Instead, it asks whether artist-level popularity/followers are associated with appearing more often in the playlist.

In [25]:
artist_count_df = (
    analysis_df.groupby(["main_artist_id", "main_artist_name"])
    .agg(
        artist_track_count=("track_id", "count"),
        artist_popularity=("artist_popularity", "max"),
        artist_followers=("artist_followers", "max"),
        avg_track_popularity=("popularity", "mean"),
    )
    .reset_index()
)
artist_count_df["log_artist_followers"] = np.log1p(artist_count_df["artist_followers"])
artist_count_df.head()

,main_artist_id,main_artist_name,artist_track_count,artist_popularity,artist_followers,avg_track_popularity,log_artist_followers
0,01HEACGPo5xyiXgAJKEvxQ,Daishi Dance,1,37,30649,54.000000,10.330388
1,02VPWD8AZ7qSuug0dM4Hk1,Chris Lee,1,41,20205,45.000000,9.913735
2,04T9gfyccsmxG79OSJp5r1,胡歌,2,33,27693,25.500000,10.228971
3,04gDigrS5kc9YWfZHwBETP,Maroon 5,3,86,47831351,53.000000,17.683192
4,07QEuhtrNmmZ0zEcqE9SF6,Owl City,3,68,2577494,47.333333,14.762329


In [26]:
poisson_model = smf.glm(
    "artist_track_count ~ artist_popularity + log_artist_followers",
    data=artist_count_df,
    family=sm.families.Poisson(),
).fit()

poisson_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:     artist_track_count   No. Observations:                  131
Model:                            GLM   Df Residuals:                      128
Model Family:                 Poisson   Df Model:                            2
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -226.53
Date:                Sat, 23 May 2026   Deviance:                       156.59
Time:                        14:32:56   Pearson chi2:                     266.
No. Iterations:                     5   Pseudo R-squ. (CS):             0.2150
Covariance Type:            nonrobust                                         
========================================================================================
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               -1.3050      0.478     -2.733      0.006      -2.241      -0.369
artist_popularity       -0.0037      0.011     -0.340      0.733      -0.025       0.018
log_artist_followers     0.1560      0.076      2.048      0.041       0.007       0.305
========================================================================================
"""

In [27]:
poisson_diagnostics = pd.Series({
    "n_artists": len(artist_count_df),
    "mean_artist_track_count": artist_count_df["artist_track_count"].mean(),
    "variance_artist_track_count": artist_count_df["artist_track_count"].var(ddof=1),
    "pearson_chi2_over_df": poisson_model.pearson_chi2 / poisson_model.df_resid,
    "deviance_over_df": poisson_model.deviance / poisson_model.df_resid,
})

poisson_diagnostics.round(3).to_frame("value")

,value
n_artists,131.000
mean_artist_track_count,1.771
variance_artist_track_count,5.178
pearson_chi2_over_df,2.077
deviance_over_df,1.223


In [28]:
poisson_plot_df = artist_count_df.copy()
poisson_plot_df["predicted_count"] = poisson_model.predict(artist_count_df)

alt.Chart(poisson_plot_df).mark_circle(size=75, opacity=0.7).encode(
    x=alt.X("predicted_count:Q", title="Predicted artist track count"),
    y=alt.Y("artist_track_count:Q", title="Observed artist track count"),
    tooltip=["main_artist_name:N", "artist_track_count:Q", "predicted_count:Q", "artist_popularity:Q", "artist_followers:Q"],
).properties(
    title="Poisson Model: Observed vs Predicted Artist Counts",
    width=650,
    height=360,
)

alt.Chart(...)

## 10. Statistical Takeaways

### 10.1 Main Findings

- The confidence intervals suggest that Latin-script songs in this playlist tend to have higher Spotify popularity than Chinese-script songs. However, this result should be interpreted cautiously. The sample sizes across language/script groups are not balanced, and this playlist is shaped by my own listening background and cultural context.

- As someone from an East Asian cultural background, the English songs I listen to may be disproportionately internationally well-known. In other words, my English-language/Latin-script tracks may be selected from songs that are already globally popular enough to be familiar across cultures. At the same time, Chinese-language music may be disadvantaged on Spotify's popularity metric because Spotify is not the dominant music platform for many mainland Chinese listeners. Therefore, this analysis reflects popularity inside Spotify's ecosystem, not necessarily global popularity or cultural value.

- Even with these limitations, the hypothesis tests provide evidence that Chinese-script and Latin-script songs in this playlist differ in popularity. The broader multi-group tests also suggest that popularity varies across language/script groups. In contrast, the release-decade tests do not provide strong evidence that popularity differs meaningfully across decades in this playlist.

### 10.2 Regression Interpretation

- The correlation analysis shows that track popularity is associated with artist popularity, although the relationship is not extremely strong. This makes practical sense: famous artists can still have less popular songs, and track-level popularity depends on more than artist-level fame.

- In the multiple linear regression model, artist popularity is the only clearly significant predictor of track popularity after controlling for release year, duration, artist followers, explicitness, and language/script group. This suggests that artist popularity is the strongest explanatory variable among the features currently available.

- However, this does not mean artist popularity fully explains track popularity. The relationship is useful but incomplete. A recommendation or prediction model should not rely only on artist popularity, because doing so would bias the system toward mainstream artists and miss niche songs that still match the user's taste.

### 10.3 Model Diagnostics and Limitations

- The OLS diagnostics show evidence of heteroskedasticity, meaning that the residual variance is not constant across fitted values. This weakens the reliability of standard OLS standard errors, so robust standard errors are more appropriate for inference.

- There is also evidence of multicollinearity, especially among artist-related variables such as artist popularity and artist followers. This is expected because more popular artists often have more followers. Multicollinearity makes individual coefficient estimates less stable, so the regression should be treated as exploratory rather than definitive.

- The logistic regression model provides a useful alternative framing by predicting whether a track is above the playlist's median popularity. This is closer to a classification problem and can serve as a bridge toward later ML modelling. However, because this is still based on a small observational playlist sample, its performance should not be interpreted as generalizable.

- The optional Poisson regression is useful for artist-level count modelling rather than track-level popularity. It asks whether more popular artists appear more frequently in the playlist. If the overdispersion diagnostics are large, a Negative Binomial model would be more appropriate than Poisson.

### 10.4 Implications for ML

- For later ML modelling, artist popularity should likely be included as an important feature, but it should not be the only feature. Language/script features may capture meaningful playlist-specific differences, although they should be interpreted with cultural and platform bias in mind. Release decade appears less useful for explaining popularity in this dataset, but it may still be useful for describing listening taste.

- The regression results also suggest that future models may benefit from regularization or models that handle correlated predictors better, such as ridge regression, random forests, or gradient boosting. For recommendation, the goal should not simply be to predict mainstream popularity. A better system should balance popularity with personal taste signals such as genre, language, artist patterns, and eventually audio features or similarity-based features.